# LangChain (open source): Models, Prompts and Output Parsers

### Outline
- Direct API call to Ollama (no LangChain yet)
- The same thing through LangChain:
    - Prompt template
    - Model
- Output parsers

Open-source recode of `02-LangChain-for-LLM-Application-Development/L1-Model_prompt_parser.ipynb`,
using Ollama's cloud API instead of OpenAI. `ResponseSchema` + `StructuredOutputParser`, the
original's output-parsing tool, were removed from modern LangChain along with the rest of the
pre-1.0 parser zoo - the replacement everywhere in this repo is
`model.with_structured_output(PydanticModel)`, used at the end of this notebook.

## Setup

This notebook is the open-source / Ollama-cloud recode of the matching lesson in
[`openai_agentic_ai_course`](../../openai_agentic_ai_course/), reusing the shared
`common.py` / `tracing.py` helpers already built for
[`open_source_agentic_ai_course`](../../open_source_agentic_ai_course/) rather than duplicating them here.

- **Model**: `ChatOllama`, pointed at the Ollama cloud endpoint configured in the
  repo-root `.env` (`OLLAMA_MODEL` / `OLLAMA_BASE_URL` / `OLLAMA_API_KEY`).
- **Tracing**: every `.invoke()` / `.batch()` / `.stream()` call below passes
  `config=traced("run name")`, which attaches a Langfuse callback - open the
  Langfuse dashboard and filter by trace name to see this notebook's calls.
- **Kernel**: run this with the repo's `.venv` (`Python 3 (ipykernel)`) - it already
  has everything in [`requirements.txt`](../../requirements.txt) installed.

In [1]:
import sys
from pathlib import Path

# common.py / tracing.py live in open_source_agentic_ai_course/, not here - add it to sys.path
# instead of copying them, so this notebook always uses the one shared implementation.
COURSE_DIR = Path("../../open_source_agentic_ai_course").resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

import os

from common import get_model, traced
from ollama import Client
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from tracing import langfuse

## Chat API: Ollama (direct)

Let's start with a direct API call to Ollama's cloud endpoint - the open-source equivalent of
the original notebook's raw `openai.ChatCompletion.create(...)` call. There's no LangChain
runnable here for Langfuse's callback handler to attach to, so this wraps the call by hand in
`langfuse.start_as_current_observation(..., as_type="generation")`.

In [2]:
client = Client(
    host=os.environ["OLLAMA_BASE_URL"],
    headers={"Authorization": f"Bearer {os.environ['OLLAMA_API_KEY']}"},
)
MODEL = os.environ["OLLAMA_MODEL"]


def get_completion(prompt: str, name: str) -> str:
    with langfuse.start_as_current_observation(
        name=name, as_type="generation", model=MODEL, input=prompt
    ) as generation:
        response = client.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
        generation.update(
            output=response.message.content,
            usage_details={"input": response.prompt_eval_count, "output": response.eval_count},
        )
        return response.message.content

In [3]:
get_completion("What is 1+1?", "L1: direct API call - 1+1")

'1\u202f+\u202f1\u202f=\u202f**2**'

Now a more realistic example: rewrite an angry, pirate-speak customer email in a calm,
respectful American-English tone.

In [5]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

style = """American English \
in a calm and respectful tone
"""

prompt = f"""Translate the text \
that is delimited by triple backticks
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

Translate the text that is delimited by triple backticks
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [6]:
response = get_completion(prompt, "L1: direct API call - style transfer")
response

'I’m frustrated that the blender lid flew off and splattered my kitchen walls with smoothie. Unfortunately, the warranty does not cover the cost of cleaning up my kitchen. I would appreciate your help with this matter.'

## Chat API: LangChain

Let's do the same thing through LangChain instead of the raw client.

### Prompt template

In [7]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""
prompt_template = ChatPromptTemplate.from_template(template_string)
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

In [8]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [9]:
customer_style = """American English \
in a calm and respectful tone
"""

customer_messages = prompt_template.format_messages(
    style=customer_style, text=customer_email
)
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [10]:
# Prompt actually sent to the model
print(customer_messages[0])

content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n" additional_kwargs={} response_metadata={}


In [11]:
model = get_model()

customer_response = model.invoke(
    customer_messages, config=traced("L1: LangChain style transfer (customer)")
)
print(customer_response.content)

I’m upset that my blender lid flew off and splattered my kitchen walls with smoothie. To make matters worse, the warranty doesn’t cover the cost of cleaning up my kitchen. I need your help right now.


Same template, different inputs - a blunt customer-service reply, translated into a polite
pirate tone.

In [12]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

service_messages = prompt_template.format_messages(
    style=service_style_pirate, text=service_reply
)
print(service_messages[0].content)

Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in English Pirate. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
```



In [13]:
service_response = model.invoke(
    service_messages, config=traced("L1: LangChain style transfer (service)")
)
print(service_response.content)

Ahoy, esteemed customer! I regret to inform ye that the warranty shall not cover the cleaning expenses for yer kitchen, as it appears the mishap arose from misusing the blender by forgetting to secure the lid before setting it in motion. Unfortunately, this be not covered. Thank ye kindly for yer understanding. Farewell!


## Output Parsers

Let's start with defining how we would like the LLM output to look:

In [14]:
{
    "gift": False,
    "delivery_days": 5,
    "price_value": "pretty affordable!",
}

{'gift': False, 'delivery_days': 5, 'price_value': 'pretty affordable!'}

In [15]:
customer_review = """\
This leaf blower is pretty amazing. It has four settings: \
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price, \
and output them as a list of strings.

text: {text}
"""

plain_prompt = ChatPromptTemplate.from_template(review_template)
plain_messages = plain_prompt.format_messages(text=customer_review)
plain_response = model.invoke(plain_messages, config=traced("L1: unstructured extraction"))
print(plain_response.content)

```json
{
  "gift": true,
  "delivery_days": 2,
  "price_value": [
    "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
  ],
  "text": "This leaf blower is pretty amazing. It has four settings: candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
}
```


In [16]:
type(plain_response.content)

str

In [17]:
# This fails: `.content` is a plain string, not a dict, so `.get()` doesn't exist on it.
plain_response.content.get('gift')

AttributeError: 'str' object has no attribute 'get'

## Parse the LLM output into a Python object

The original notebook does this with `ResponseSchema` + `StructuredOutputParser`, which built a
format-instructions string to stuff into the prompt and then hand-parsed the reply. Modern
LangChain replaces that whole flow with one call: give the model a Pydantic schema via
`with_structured_output(...)` and get back a validated instance directly - no separate parser,
no format-instructions string to write.

In [19]:
class ReviewInfo(BaseModel):
    """Information extracted from a product review."""

    gift: bool = Field(description="Was the item purchased as a gift for someone else?")
    delivery_days: int = Field(description="How many days it took to arrive. -1 if not found.")
    price_value: list[str] = Field(description="Sentences about the value or price of the item.")


structured_model = model.with_structured_output(ReviewInfo, method="function_calling")
structured_chain = plain_prompt | structured_model

result = structured_chain.invoke(
    {"text": customer_review}, config=traced("L1: structured extraction")
)
result

ReviewInfo(gift=True, delivery_days=2, price_value=["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."])

In [20]:
type(result)

__main__.ReviewInfo

In [21]:
result.delivery_days

2